In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paper result-package finalizer

Thin engineering-canary entry. It verifies Drive waiting-state publication without creating a scientific result package.


In [ ]:
import json, os, pathlib, subprocess, sys

REPO='https://github.com/RICHAAARC/CEG-WM.git'
EXPECTED_EXACT='9ec454055c74cf4ed89001387c9f700e9ba5aef0'
BASELINE_EXACT='e4cf4ed2738cb91204695efbf9fb6ce35858b5f7'
checkout=pathlib.Path('/content/cegwm-paper-finalize')
drive_root=pathlib.Path('/content/drive/MyDrive/CEG-WM/PaperFormal-V1-EngineeringCanary/finalizer')
if not checkout.exists(): subprocess.run(['git','clone',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
head=subprocess.run(['git','-C',str(checkout),'rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip()
dirty=subprocess.run(['git','-C',str(checkout),'status','--porcelain'],check=True,capture_output=True,text=True).stdout.strip()
assert head==EXPECTED_EXACT and not dirty
child_env=dict(os.environ)
child_env['PYTHONPATH']=str(checkout/'src')+os.pathsep+str(checkout)
command=[sys.executable,'-m','experiments.run_paper_results_finalize','--drive-root',str(drive_root),'--expected-exact',EXPECTED_EXACT,'--baseline-exact',BASELINE_EXACT,'--engineering-canary']
subprocess.run(command,cwd=checkout,env=child_env,check=True)
final_path=drive_root/'finalized'/'paper-formal-v1'/'canary_final.json'
public=json.loads(final_path.read_text(encoding='utf-8'))
print({'status':public['status'],'science_denominator':public['science_denominator'],'premature_final_absent':public['premature_final_absent']})
